# C-GDP and AA-GDP: Numerical Equivalence Check

**Purpose.** Validate the implementation of the collaboration-matrix identity in Proposition III.1. This is a synthetic, float64 numerical check, not a statistical equivalence test, a privacy evaluation, or a utility experiment. No image dataset, classifier, GPU, repository checkout, or pretrained model is required.

Run the cells in order. The experiment runs on the Colab CPU. The final optional cell saves the latest results into `MyDrive/Noisy Anchor Alignment/results/equivalence`; running it again replaces those result files. The notebook itself is the complete standalone source.

For participant $i$, construct $Y_i=X_iO_i+\mathbf1\psi_i^\top+\sigma W_i$ and **noiseless** anchors $B_i=AO_i+\mathbf1\psi_i^\top$. AA-GDP estimates $R_i$ only from the centered anchor cross-product using an SVD and returns $(Y_i-\mathbf1b_i^\top)R_i+\mathbf1b_s^\top$.

Independently, C-GDP uses $X_iO_s+\mathbf1\psi_s^\top+\sigma\widetilde W_i$, where $\widetilde W_i=W_iO_i^\top O_s$. The analytic rotation $O_i^\top O_s$ is used only to construct this prescribed coupling and to score the estimated rotation; it is never supplied to AA-GDP alignment.

The Gaussian noise has the same law after orthogonal rotation. Coupling makes the two realizations equal; independent draws would agree in distribution without agreeing numerically.


## Design and acceptance criteria

- Core: $d\in\{10,50,100,400\}$, $p\in\{2,10,50\}$, ten seeds, $r=4d$, centered orthonormal anchors: **120 deployments**.
- Additional full-rank case: the same four dimensions and ten seeds, $p=10$, $r=d+1$, a nonzero anchor centroid, and centered singular values from 1 to 0.1 (condition number 10): **40 deployments**.
- Each deployment uses $\sigma\in\{0,0.01,0.1,1,10\}$: **800 paired comparisons**. Noise levels within a deployment reuse the same standard Gaussian noise and are not independent deployments.
- Participant record counts are unequal (24, 32, or 40). Private matrices and translations are standard Gaussian; orthogonal matrices are generated by sign-corrected Gaussian QR. The reference participant varies with the seed.
- Primary residual: $e_Z=\|Z_{\rm AA}-Z_{\rm C}\|_F/\|Z_{\rm C}\|_F$. Also retain the largest participant-level relative error, normalized largest entrywise discrepancy, and $e_R=\max_i\|R_i-O_i^\top O_s\|_F/\sqrt d$.
- Prespecified float64 tolerance: $10^{-10}$ for all scored residuals. Record failures without discarding configurations. No p-values, confidence intervals, or statistical equivalence margins are used.
- Negative control: replace the C-GDP coupled noise with independent Gaussian noise at nonzero $\sigma$. Its discrepancy is reported as a sensitivity check; nonzero discrepancy does not contradict distributional equivalence.

The full-rank cases are deliberately well conditioned. Results do not assert numerical stability for arbitrarily ill-conditioned anchors.


In [ ]:
import json
import math
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_info, threadpool_limits

DIMENSIONS = (10, 50, 100, 400)
PARTICIPANTS = (2, 10, 50)
SIGMAS = (0.0, 0.01, 0.1, 1.0, 10.0)
SEEDS = tuple(range(10))
MASTER_SEED = 20260906
TOLERANCE = 1e-10
LOCAL_RESULTS = Path('/content/equivalence_results')
LOCAL_RESULTS.mkdir(exist_ok=True, parents=True)

def orthogonal(rng, d):
    q, r = np.linalg.qr(rng.standard_normal((d, d)))
    signs = np.where(np.diag(r) < 0.0, -1.0, 1.0)
    return q * signs

def make_anchor(rng, d, profile):
    r = 4 * d if profile == 'orthonormal' else d + 1
    a = rng.standard_normal((r, d))
    a -= a.mean(axis=0)
    u, _ = np.linalg.qr(a, mode='reduced')
    if profile == 'orthonormal':
        return u
    singular_values = np.geomspace(1.0, 0.1, d)
    return (u * singular_values) @ orthogonal(rng, d).T + rng.standard_normal(d)

def aa_alignment(anchor_uploads, reference):
    # This function only sees uploaded anchors, not A, X, or secret rotations.
    means = [b.mean(axis=0) for b in anchor_uploads]
    centered = [b - mean for b, mean in zip(anchor_uploads, means)]
    d = centered[0].shape[1]
    rotations = []
    for i, b in enumerate(centered):
        if i == reference:
            rotations.append(np.eye(d))
        else:
            u, _, vt = np.linalg.svd(b.T @ centered[reference], full_matrices=False)
            rotations.append(u @ vt)
    return rotations, means

def relative_frobenius(a, b):
    return float(np.linalg.norm(a - b, 'fro') / max(np.linalg.norm(b, 'fro'), np.finfo(float).tiny))

def run_deployment(d, p, seed, profile):
    profile_number = 0 if profile == 'orthonormal' else 1
    rng = np.random.default_rng(np.random.SeedSequence([MASTER_SEED, d, p, seed, profile_number]))
    a = make_anchor(rng, d, profile)
    a_centered = a - a.mean(axis=0)
    spectrum = np.linalg.svd(a_centered, compute_uv=False)
    rank = int(np.linalg.matrix_rank(a_centered))
    assert rank == d, (d, p, seed, profile, rank)
    os = [orthogonal(rng, d) for _ in range(p)]
    shifts = [rng.standard_normal(d) for _ in range(p)]
    xs = [rng.standard_normal((24 + 8 * (i % 3), d)) for i in range(p)]
    ws = [rng.standard_normal(x.shape) for x in xs]
    independent_ws = [rng.standard_normal(x.shape) for x in xs]
    s = seed % p
    bs = [a @ o + shift for o, shift in zip(os, shifts)]
    estimated_rs, means = aa_alignment(bs, s)
    analytic_rs = [o.T @ os[s] for o in os]
    rotation_error = max(relative_frobenius(rhat, rexact) for rhat, rexact in zip(estimated_rs, analytic_rs))
    orthogonality_error = max(relative_frobenius(rhat.T @ rhat, np.eye(d)) for rhat in estimated_rs)
    # These separate constructions must not be replaced by copies of AA output.
    private_bases = [x @ o + shift for x, o, shift in zip(xs, os, shifts)]
    common_bases = [x @ os[s] + shifts[s] for x in xs]
    coupled_ws = [w @ rtrue for w, rtrue in zip(ws, analytic_rs)]
    output = []
    for sigma in SIGMAS:
        aa = [(base + sigma * w - mean) @ rhat + means[s]
              for base, w, mean, rhat in zip(private_bases, ws, means, estimated_rs)]
        cg = [base + sigma * w for base, w in zip(common_bases, coupled_ws)]
        zaa, zcg = np.vstack(aa), np.vstack(cg)
        ez = relative_frobenius(zaa, zcg)
        block_error = max(relative_frobenius(aai, cgi) for aai, cgi in zip(aa, cg))
        entry_error = float(np.max(np.abs(zaa - zcg)) / max(1.0, np.max(np.abs(zcg))))
        if sigma > 0:
            independent_cg = np.vstack([base + sigma * w for base, w in zip(common_bases, independent_ws)])
            negative_error = relative_frobenius(zaa, independent_cg)
        else:
            negative_error = float('nan')
        output.append(dict(profile=profile, dimension=d, participants=p, anchor_rows=len(a),
                           seed=seed, reference=s, sigma=sigma, records=len(zaa),
                           anchor_rank=rank, anchor_condition=float(spectrum[0] / spectrum[-1]),
                           anchor_centroid_norm=float(np.linalg.norm(a.mean(axis=0))),
                           rotation_error=rotation_error, orthogonality_error=orthogonality_error,
                           matrix_error=ez, max_block_error=block_error, entry_error=entry_error,
                           independent_noise_error=negative_error,
                           passed=bool(np.isfinite([ez, block_error, entry_error, rotation_error, orthogonality_error]).all()
                                       and max(ez, block_error, entry_error, rotation_error, orthogonality_error) <= TOLERANCE)))
    return output

print('Ready: float64 CPU experiment; 160 deployments and 800 paired comparisons.')


In [ ]:
started = datetime.now(timezone.utc).isoformat()
tic = time.perf_counter()
rows = []
configs = [(d, p, seed, 'orthonormal') for d in DIMENSIONS for p in PARTICIPANTS for seed in SEEDS]
configs += [(d, 10, seed, 'minimal_full_rank') for d in DIMENSIONS for seed in SEEDS]
with threadpool_limits(limits=2):
    numerical_libraries = threadpool_info()
    for k, (d, p, seed, profile) in enumerate(configs, 1):
        rows.extend(run_deployment(d, p, seed, profile))
        if k % 10 == 0:
            pd.DataFrame(rows).to_csv(LOCAL_RESULTS / 'equivalence_raw.csv', index=False)
            print(f'{k}/{len(configs)} deployments; d={d}, p={p}, {profile}; {time.perf_counter()-tic:.1f}s', flush=True)
raw = pd.DataFrame(rows)
summary = raw.groupby(['profile', 'dimension'], sort=False).agg(
    comparisons=('passed', 'size'), max_matrix_error=('matrix_error', 'max'),
    max_block_error=('max_block_error', 'max'), max_rotation_error=('rotation_error', 'max'),
    max_entry_error=('entry_error', 'max'), passed=('passed', 'sum')).reset_index()
manifest = dict(experiment='C-GDP / AA-GDP numerical equivalence',
                source_notebook='C_GDP_AA_GDP_Equivalence.ipynb', started_utc=started,
                finished_utc=datetime.now(timezone.utc).isoformat(), elapsed_seconds=time.perf_counter()-tic,
                python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                platform=platform.platform(), numerical_libraries=numerical_libraries,
                dtype='float64', master_seed=MASTER_SEED, seeds=list(SEEDS),
                dimensions=list(DIMENSIONS), core_participants=list(PARTICIPANTS), sigmas=list(SIGMAS),
                deployments=len(configs), paired_comparisons=len(raw), tolerance=TOLERANCE,
                all_passed=bool(raw.passed.all()), max_matrix_error=float(raw.matrix_error.max()),
                max_block_error=float(raw.max_block_error.max()), max_rotation_error=float(raw.rotation_error.max()),
                max_entry_error=float(raw.entry_error.max()), max_orthogonality_error=float(raw.orthogonality_error.max()),
                independent_noise_error_min=float(raw.independent_noise_error.min()),
                independent_noise_error_max=float(raw.independent_noise_error.max()),
                statistical_claim=False)
raw.to_csv(LOCAL_RESULTS / 'equivalence_raw.csv', index=False)
summary.to_csv(LOCAL_RESULTS / 'equivalence_summary.csv', index=False)
(LOCAL_RESULTS / 'equivalence_manifest.json').write_text(json.dumps(manifest, indent=2))
print('\nSUMMARY (maximum residuals, not confidence intervals)')
display(summary.style.format({c: '{:.3e}' for c in summary if 'error' in c}))
print('\nMANIFEST_JSON\n' + json.dumps(manifest, indent=2))
assert len(raw) == 800 and raw.passed.all(), 'A numerical check failed. Inspect the saved rows; do not omit failures.'
print('\nALL 800 PAIRED COMPARISONS PASSED THE PRESPECIFIED NUMERICAL TOLERANCE.')


In [ ]:
scored_columns = ['matrix_error', 'max_block_error', 'entry_error', 'rotation_error', 'orthogonality_error']
assert len(raw) == 800 and np.isfinite(raw[scored_columns].to_numpy()).all() and raw.passed.all(), 'Do not generate a success report for failed or incomplete results.'
table_lines = ['| Anchor configuration | Dimension | Comparisons | Maximum matrix residual | Maximum block residual | Maximum rotation residual |',
               '|---|---:|---:|---:|---:|---:|']
for row in summary.itertuples():
    table_lines.append(f'| {row.profile} | {row.dimension} | {row.comparisons} | {row.max_matrix_error:.3e} | {row.max_block_error:.3e} | {row.max_rotation_error:.3e} |')
claim = (f'In float64 synthetic implementation checks spanning dimensions 10, 50, 100, and 400, '
         f'participant counts 2, 10, and 50, and private-data-noise scales 0, 0.01, 0.1, 1, and 10, '
         f'the 800 paired C-GDP and AA-GDP collaboration matrices agreed to a maximum relative Frobenius '
         f'discrepancy of {raw.matrix_error.max():.3e}. This includes 600 comparisons with centered orthonormal '
         f'anchors (r=4d) and 200 additional comparisons with 10 participants and shifted full-rank anchors '
         f'at r=d+1 whose centered condition number is 10. All scored residuals were below the prespecified '
         f'10^-10 numerical tolerance. These checks validate the implementation of Proposition III.1 '
         f'under its prescribed noise coupling; they are not a statistical test of equivalence.')
report = '# Numerical validation of C-GDP / AA-GDP equivalence\n\n' + claim + '\n\n' + '\n'.join(table_lines)
report += ('\n\nThe 800 comparisons share five noise levels within each of 160 deployments. '
           'They must not be described as 800 independent deployments. No image data, labels, model, '
           'accuracy measure, linkage auditor, or privacy claim is used. The tolerance and grid were specified before execution. '
           'The uncoupled-noise control produces different matrices while preserving the same marginal Gaussian law.\n')
(LOCAL_RESULTS / 'equivalence_report.md').write_text(report)
latex = [r'\begin{tabular}{llrrr}', r'\hline',
         r'Anchor & $d$ & Comparisons & Max. $e_Z$ & Max. $e_R$ \\', r'\hline']
for row in summary.itertuples():
    label = 'Orthonormal' if row.profile == 'orthonormal' else 'Full-rank, minimal'
    latex.append(f'{label} & {row.dimension} & {row.comparisons} & {row.max_matrix_error:.2e} & {row.max_rotation_error:.2e} ' + r'\\')
latex.extend([r'\hline', r'\end{tabular}'])
(LOCAL_RESULTS / 'equivalence_table.tex').write_text('\n'.join(latex) + '\n')
print(claim)
print('\nResults written to:', LOCAL_RESULTS)


## Save the latest results to the project folder

This cell requests your usual Google Drive mount if needed, then writes only the five named result files to `Noisy Anchor Alignment/results/equivalence`. It does not modify the old NAA-GDP repository. The notebook and a short README are the only source files required in the new project root.


In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
destination = Path('/content/drive/MyDrive/Noisy Anchor Alignment/results/equivalence')
destination.mkdir(parents=True, exist_ok=True)
for name in ('equivalence_raw.csv', 'equivalence_summary.csv', 'equivalence_manifest.json',
             'equivalence_report.md', 'equivalence_table.tex'):
    shutil.copy2(LOCAL_RESULTS / name, destination / name)
print('Saved latest results to:', destination)
print('\n'.join(sorted(p.name for p in destination.iterdir() if p.is_file())))
